In [2]:
!pip install ccxt boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 11.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 11.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [boto3]m13/15 [ccxt]tp]]


In [ ]:
import ccxt
import pandas as pd
import time
import io
import boto3
from botocore.client import Config

MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "password123"
BUCKET_NAME = "crypto-raw-data"

# 1. Kết nối vào kho MinIO
s3_client = boto3.client(
    's3',
    endpoint_url="http://minio:9000", 
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4')
)

try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"ℹ️ Bucket '{BUCKET_NAME}' đã tồn tại.")
except:
    s3_client.create_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Đã tự động tạo mới Bucket: '{BUCKET_NAME}' trên MinIO.")

exchange = ccxt.bitstamp()

altcoins = [
    'ETH/USD', 'XRP/USD', 'LTC/USD', 'LINK/USD', 'UNI/USD',
    'MATIC/USD', 'SOL/USD', 'ADA/USD', 'DOT/USD', 'AVAX/USD',
    'DOGE/USD', 'SHIB/USD', 'BCH/USD', 'ALGO/USD', 'AAVE/USD'
]
df_alts_list = []

print("\nBắt đầu tải dữ liệu Altcoins (500 ngày)...")
for coin in altcoins:
    try:
        ohlcv = exchange.fetch_ohlcv(coin, '1d', limit=500)
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df['symbol'] = coin 
        
        df_alts_list.append(df)
        print(f"  -> Đã tải xong {coin}")
        time.sleep(1) 
    except Exception as e:
        print(f"  ❌ Lỗi khi tải {coin}: {e}")

# 3. Đẩy file Altcoins lên MinIO
if df_alts_list:
    df_all_alts = pd.concat(df_alts_list, ignore_index=True)
    df_all_alts = df_all_alts[['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume']]
    
    alt_buffer = io.StringIO()
    df_all_alts.to_csv(alt_buffer, index=False)
    s3_client.put_object(Bucket=BUCKET_NAME, Key='altcoins_500d.csv', Body=alt_buffer.getvalue())
    print(f"\n🚀 Đạt yêu cầu: Đã đẩy thành công 'altcoins_500d.csv' ({len(df_all_alts)} dòng) lên MinIO!")

In [2]:
!pip install s3fs


In [3]:
import pandas as pd
import s3fs
from pyspark.sql import SparkSession

# 1. Khởi tạo Spark Session không cần config S3A rắc rối
spark = SparkSession.builder \
    .appName("Realtime_Native_Read") \
    .getOrCreate()

# 2. Dùng s3fs để đọc thẳng từ MinIO qua giao thức S3 (Không phải S3A)
fs = s3fs.S3FileSystem(
    key='admin',
    secret='password123',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

# 3. Đọc dữ liệu vào Pandas trước (dùng s3fs), sau đó convert sang Spark DataFrame
# Cách này cực nhanh và bỏ qua hoàn toàn lớp Java/Hadoop
with fs.open('crypto-raw-data/bitcoin_1m.csv', 'rb') as f:
    pdf = pd.read_csv(f)
    df = spark.createDataFrame(pdf)

print("✅ Đã đọc dữ liệu thành công qua s3fs!")
df.show(5)

✅ Đã đọc dữ liệu thành công qua s3fs!
+-------------------+--------+--------+--------+--------+-----------+
|          timestamp|    open|    high|     low|   close|     volume|
+-------------------+--------+--------+--------+--------+-----------+
|2026-06-06 00:07:00| 61207.2|61263.22|61197.13|61263.22| 2.31223182|
|2026-06-06 00:08:00|61263.22|61373.54|61231.79|61357.05| 3.98910434|
|2026-06-06 00:09:00|61378.41|61406.16|61304.74|61324.18| 3.06868813|
|2026-06-06 00:10:00| 61318.4| 61318.4|61195.16|61211.92| 1.64304321|
|2026-06-06 00:11:00|61211.44|61245.89|60933.95|60940.71|11.06868633|
+-------------------+--------+--------+--------+--------+-----------+
only showing top 5 rows
